In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import insightface
import random
from datetime import datetime

In [ ]:
path_sabrina = "/mnt/nas2/sabrina/face-gen/embeddings_sabrina.pkl"
#path = "/mnt/nas2/sabrina/face-gen/embeddings-0312.pkl"
df = pd.read_pickle(path_sabrina)

print(type(df['embedding'].iloc[0]))  # <class 'numpy.ndarray'>
is_array_col = df['embedding'].apply(lambda x: isinstance(x, np.ndarray))
print(is_array_col.value_counts())

# convert back to float32
df['embedding'] = df['embedding'].apply(lambda x: x.astype(np.float32))

<class 'numpy.ndarray'>
embedding
True    13199
Name: count, dtype: int64


In [3]:
df.head()

,name,ID,embedding,embedding type,embedding dtype,embedding_size,image_name,filepath,embedding_type,embedding_dtype
0,AJ_Cook,0,"[-0.3419863, 0.6712041, -1.2866051, 0.60350394...",<class 'numpy.ndarray'>,float32,512,AJ_Cook_0001.jpg,/home/sho/Insightface-Face-Recognition/img/LFW...,NaN,NaN
1,AJ_Lamas,1,"[0.059667192, 0.076293096, 1.3340704, 1.245711...",<class 'numpy.ndarray'>,float32,512,AJ_Lamas_0001.jpg,/home/sho/Insightface-Face-Recognition/img/LFW...,NaN,NaN
2,Aaron_Eckhart,2,"[1.2447274, -0.9747005, 1.2987878, 0.46920043,...",<class 'numpy.ndarray'>,float32,512,Aaron_Eckhart_0001.jpg,/home/sho/Insightface-Face-Recognition/img/LFW...,NaN,NaN
3,Aaron_Guiel,3,"[-0.17151171, -1.2514898, -0.38039768, -0.0818...",<class 'numpy.ndarray'>,float32,512,Aaron_Guiel_0001.jpg,/home/sho/Insightface-Face-Recognition/img/LFW...,NaN,NaN
4,Aaron_Patterson,4,"[0.41729146, -0.24729499, -0.86146384, -2.1820...",<class 'numpy.ndarray'>,float32,512,Aaron_Patterson_0001.jpg,/home/sho/Insightface-Face-Recognition/img/LFW...,NaN,NaN


In [4]:
df['embedding'].iloc[0].dtype

dtype('float32')

# fvs

In [15]:
# fvs search
import fvs_face_helper as fvs
import importlib
importlib.reload(fvs)

def fvs_search(df, test_mode=False):
    top_1_pos, top_3_pos, top_10_pos = 0, 0, 0
    total = 0

    top_1_failures, top_3_failures, top_10_failures = [], [], []
    inner_search_time = []
    outer_search_time = []

    if test_mode:
        print("##### test mode #######")

    for query_idx in tqdm(range(len(df)), desc="Evaluating search accuracy"):
        if test_mode:
            if query_idx >= 100:
                break
        
        query_person_id = df.iloc[query_idx]['ID']
        query_name = df.iloc[query_idx]['name']
        query_emb = df['embedding'].iloc[query_idx]
        #print("query idx:", query_idx)
        
        # Skip if only one image for this person
        if df[df.ID == query_person_id].shape[0] == 1:
            continue
        
        total += 1
        #print("query idx:", query_idx)

        similarities = []

        topk = 10
        query_emb = query_emb.reshape((1, 512))
        
        start_time = datetime.now()
        response, inner_duration = fvs.face_search("d4b33d5c-0719-411b-ace4-8b5ff8b4004c", query_emb, topk, verbose=False )
        #print("response=", response)
        end_time = datetime.now()
        outer_duration = (end_time - start_time).total_seconds()
        inner_search_time.append(inner_duration)
        outer_search_time.append(outer_duration)

        # slice idx 1:10 (ignore top 1)
        top_k_indices = response.indices[0][1:] # 9
        top_k_scores = response.distance[0][1:]
        top_k_scores = [float(x) for x in top_k_scores]
    
        #print("top k indicies:", top_k_indices)
        #print("shape:", df.shape)
        top_k_ids = [int(df.iloc[i]['ID']) for i in top_k_indices]
        


        ## Top-1 evaluation
        if df.iloc[top_k_indices[0]]['ID'] == query_person_id:
            #print("top 1 id lst:", df.iloc[top_k_indices[0]]['ID'])
            #print("top 1 person id: ", query_person_id)
            top_1_pos += 1
        else:
            top_1_failures.append({
                'query_idx': query_idx,
                'query_id': query_person_id,
                'query_name': query_name,
                'top_k_indices': top_k_indices[:1], 
                'top_k_scores': top_k_scores[:1], 
                'top_k_names': [df.iloc[i]['name'] for i in top_k_indices[:1]]
            })

        # 768

        ## Top-3 evaluation
        lst = [df.iloc[i]['ID'] for i in top_k_indices]
        #print("top 3:", lst)
        #print("top 3 person query", query_person_id)
        if query_person_id in lst[:3]:
            top_3_pos += 1
        else:
            top_3_failures.append({
                'query_idx': query_idx,
                'query_id': query_person_id,
                'query_name': query_name,
                'top_k_indices': top_k_indices[:3],
                'top_k_ids': top_k_ids[:3],
                'top_k_scores': top_k_scores[:3], 
                'top_k_names': [df.iloc[i]['name'] for i in top_k_indices[:3]]
            })

        ## Top-10 evaluation
        if query_person_id in lst[:10]:
            top_10_pos += 1
        else:
            top_10_failures.append({
                'query_idx': query_idx,
                'query_id': query_person_id,
                'query_name': query_name,
                'top_k_indices': top_k_indices,
                'top_k_ids': top_k_ids,
                'top_k_scores': top_k_scores, 
                'top_k_names': [df.iloc[i]['name'] for i in top_k_indices]
            })

    # === Final report ===
    top_1_acc = top_1_pos/total
    top_3_acc = top_3_pos/total    
    top_10_acc = top_10_pos/total
    inner_latency = np.median(inner_search_time) * 1000
    outer_latency = np.median(outer_search_time) * 1000

    print("Total query attempts:", total)
    print(f"Top-1 accuracy: {top_1_pos}/{total} = {top_1_pos/total:.4f}")
    print(f"Top-3 accuracy: {top_3_pos}/{total} = {top_3_pos/total:.4f}")
    print(f"Top-10 accuracy: {top_10_pos}/{total} = {top_10_pos/total:.4f}")
    print(f"Top-1 failures: {len(top_1_failures)}")
    print(f"Top-3 failures: {len(top_3_failures)}")
    print(f"Top-10 failures: {len(top_10_failures)}")
    print(f"Median Inner Search Time FVS: {inner_latency}")
    print(f"Median Outer Search Time FVS: {outer_latency}")

    return top_k_indices, top_k_scores, top_1_acc, top_3_acc, top_10_acc, inner_latency, outer_latency


In [17]:
top_k_indices, top_k_scores, top_1_acc, top_3_acc, top_10_acc, inner_latency, outer_latency = fvs_search(df, test_mode=False)

Evaluating search accuracy: 100%|██████████| 13199/13199 [05:07<00:00, 42.91it/s] 

Total query attempts: 9137
Top-1 accuracy: 8929/9137 = 0.9772
Top-3 accuracy: 9001/9137 = 0.9851
Top-10 accuracy: 9004/9137 = 0.9854
Top-1 failures: 208
Top-3 failures: 136
Top-10 failures: 133
Median Inner Search Time FVS: 8.226
Median Outer Search Time FVS: 29.63
